## Tools

Models can request to call **tools** that perform tasks such as fetching data, running calculations, or interacting with external APIs.

A tool is a pairing of:
- **A Schema**: The tool's name, description, and input/output specifications (often JSON Schema), derived from the function's name, docstring, and type hints.
- **An Implementation**: The actual Python function that executes when the tool is called.

Key methods in this notebook:
- **`@tool`**: Decorator that converts a Python function into a LangChain tool object (providing `.invoke()`).
- **`model.bind_tools([...])`**: Attaches tool schemas to a chat model so it knows which tools it can request.

In [1]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# Load environment variables and initialize the base chat model
load_dotenv()
model = init_chat_model("groq:qwen/qwen3.8-27b")

# Verify the model works with a normal text prompt
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='Parrots "talk" because they have evolved to be highly social animals that rely heavily on **vocal communication** to bond with their flock, establish territory, and signal danger. In the wild, parrots do not mimic human speech; instead, they mimic the sounds of their environment, including other birds, predators, and eventually, humans.\n\nHere’s a breakdown of why and how parrots learn to mimic human speech:\n\n### 1. Social Bonding\nParrots are flock animals. In the wild, they constantly chatter, call, and make noise to maintain social cohesion. When a parrot is kept as a pet, it views its human owner as part of its flock. Mimicking human speech is a way for the parrot to:\n- **Integrate** with its "flock" (the humans).\n- **Seek attention** or interaction.\n- **Express emotions** like excitement, frustration, or joy.\n\n### 2. Innate Ability to Mimic\nParrots, especially species like African Greys, Amazons, Cockatoos, and Budgerigars, have a unique anatomical and 

In [19]:
from langchain.tools import tool

# Using @tool creates a LangChain StructuredTool with schema + .invoke() support
@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location."""
    return f"The current weather in {location} is sunny with a temperature of 25°C."

# Plain Python functions can also be bound; LangChain infers their schema from type hints and docstring
def get_temperature(location: str) -> str:
    """Get the current temperature for a given location."""
    return f"The current temperature in {location} is 25°C."

# bind_tools() returns a new runnable that sends tool schemas with every model call
model_with_tools = model.bind_tools([get_weather, get_temperature])

In [22]:
# Ask a question that requires a tool; the model responds with tool_calls instead of a final answer
response = model_with_tools.invoke("What is the weather like in New York?")
print(response)

# Inspect each requested tool call's name and parsed arguments
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'tool_calls': [{'id': 'p5a6wsvgm', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 340, 'total_tokens': 367, 'completion_time': 0.074419052, 'completion_tokens_details': None, 'prompt_time': 0.02514466, 'prompt_tokens_details': None, 'queue_time': 0.268539021, 'total_time': 0.099563712}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a20219-2495-7238-92a8-75582e98e9db-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'p5a6wsvgm', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 340, 'output_tokens': 27, 'total_tokens': 367}
Tool: get_weather
Args: {'location': 'New York'}


### Tool Execution Loops

Binding tools only lets the model *request* a tool call—it does not execute the Python function automatically. A manual tool execution loop works in three steps:
1. Invoke `model_with_tools` with the user message to get an `AIMessage` containing `tool_calls`.
2. Execute each requested tool (e.g., `get_weather.invoke(tool_call)`) to produce a `ToolMessage` result.
3. Append both the `AIMessage` and the `ToolMessage` to `messages` and invoke the model again so it can generate the final natural-language answer.

In [23]:
# Step 1: Model generates tool_calls based on the user's question
messages = [{"role": "user", "content": "What is the weather like in New York?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)  # Keep the AI's tool-call request in the conversation history

# Step 2: Execute each tool call and append the resulting ToolMessage to messages
for tool_call in ai_msg.tool_calls:
    # Passing the tool_call dict to .invoke() automatically returns a ToolMessage with matching tool_call_id
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass the updated conversation (including tool output) back to the model for the final answer
final_response = model_with_tools.invoke(messages)
print(final_response.text)

It's currently sunny in New York with a temperature of 25°C (77°F).
